# Downstream hydrological calibration and validation

Use observed flow at White Haven (01447800) as the upstream boundary. HydroCNHS adds intervening runoff and routes it to Lehighton (01449000).

Steps: configuration → observations → lower watershed inputs → check → fixed parameter run or calibration → independent validation → saved outputs.

Default cells inspect configuration and input readiness only. Downloads and native hydrology require the explicit switches below. This notebook never constructs an LLM.

From the repository root, install the geometry dependencies before checking a complete model:

```bash
python -m pip install -e ".[gis]" -c environment/requirements-bundled.txt
```


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file() and (p / "examples/fe_walter").is_dir())
CASE = ROOT / "examples/fe_walter/downstream"
STUDY = Path(os.environ.get("HYDROREOPLLM_STUDY_DIR", str(Path.home() / "HydroReOpLLM-data/downstream-validation"))).resolve()
STUDY.mkdir(parents=True, exist_ok=True)
for name in ("validation.yaml", "calibration.yaml"):
    target = STUDY / name
    if not target.exists():
        shutil.copyfile(CASE / name, target)  # Keep later user edits.
SETTINGS = STUDY / "validation.yaml"
OBSERVATIONS = STUDY / "observations-history"
print("Study directory:", STUDY)


## 1. Read the complete configuration

The YAML file connects the two gauge IDs, four periods, observation table, lower native model and weather. Paths in your study copy are relative to that file. Fill the three lower-watershed input references after completing step 3. Units: discharge cfs in the observation table; precipitation mm/day, temperature degrees C and PET cm/day in the forcing table.

In [ ]:
print(SETTINGS.read_text(encoding="utf-8"))

## 2. Preserve historical observations

Warm-up is 1989, calibration 1990–2010 and independent validation 2011–2020. Keep 2021–2025 for the later operation comparison. Preserve the raw USGS response and its manifest alongside the normalized CSV. A,e denotes approved estimated data; the configuration states whether to include it.

Reuse an existing matching observation directory. The preparation command refuses to overwrite one. Set the switch to True only when a new download is required.

In [ ]:
DOWNLOAD_OBSERVATIONS = False

if OBSERVATIONS.exists():
    print("Reusing observations:", OBSERVATIONS)
elif DOWNLOAD_OBSERVATIONS:
    subprocess.run([sys.executable, str(CASE / "prepare_observations.py"),
                    "--output", str(OBSERVATIONS)], check=True, cwd=ROOT)
else:
    print("Observation preparation is disabled. Supply the saved directory or enable the explicit download.")


## 3. Prepare the lower native model and weather

Use the watershed between White Haven and Lehighton. The native model contains the intervening GWLF units and their Lohmann connections. Both gauges are computational outlets without local runoff; observed White Haven flow replaces the upstream outlet's assigned flow.

Required preparation:

- Delineate Lehighton basin minus White Haven basin; match every native unit ID and area, preserving the outlet precision statement.
- Prepare daily area-weighted precipitation and temperature for each unit, and derive Hamon PET consistently with the upstream study.
- Derive local soil/land attributes and routing lengths, and document how accepted parameters are transferred.
- Supply the partition JSON described in the bilingual reference. The check rejects gaps, overlaps and repeated flow contributions.

The FE Walter study uses nine local units, with complete daily weather, local physical attributes and a checked native model preserved in its research evidence. The [parameterization and regionalization reference](../examples/fe_walter/downstream/validation.md#parameterization-and-regionalization) explains how those inputs are prepared. Use your own study paths in the configuration above; the map precipitation layer provides spatial context rather than a daily forcing table.

## 4. Check inputs without hydrological execution

The check reports each missing input. It does not interpolate observations or replace missing lower runoff with an upstream/unit-area approximation.

In [ ]:
spec = importlib.util.spec_from_file_location("downstream_validation_case", CASE / "validate.py")
validation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validation)
obs_file = OBSERVATIONS / "observations.csv"
manifest = OBSERVATIONS / "source_manifest.json"
observation_arguments = []
if obs_file.is_file():
    observation_arguments += ["--observations", str(obs_file)]
if manifest.is_file():
    observation_arguments += ["--source-manifest", str(manifest)]
report, context = validation.preflight(SETTINGS, obs_file if obs_file.is_file() else None,
                                      manifest if manifest.is_file() else None)
print(json.dumps({key: report[key] for key in ("status", "expected_history_days", "blockers")}, indent=2))


## 5. Check transferred parameters or calibrate the lower model

First inspect a fixed parameter run. If lower parameters need calibration, document their groups and bounds in the calibration file. Set assigns a common value to listed native parameter paths; scale applies a shared multiplier to their initial values. An optional clip interval limits scaled values to their physical range; several units can reach the same limit.

Fill the population and generation settings explicitly. The native GA_DEAP search uses calibration-period KGE (2009), with one CPU worker. It simulates warm-up through the end of calibration only. The selected model is saved as a new file before the independent validation run.

The selector receives positive 1 / (2 - KGE) scores, preserving the ordering of raw KGE. Candidate records retain both; native optimizer summaries show the transformed selection score. Final hydrological reports show raw KGE.


In [ ]:
print((STUDY / "calibration.yaml").read_text(encoding="utf-8"))

In [ ]:
RUN_FIXED_PARAMETERS = False
RUN_CALIBRATION = False
if RUN_FIXED_PARAMETERS and RUN_CALIBRATION:
    raise ValueError("Select one action for this output directory.")

RUN_DIR = STUDY / ("run-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
command = [sys.executable, str(CASE / "validate.py"),
           "calibrate" if RUN_CALIBRATION else "run",
           "--settings", str(SETTINGS), *observation_arguments, "--output", str(RUN_DIR)]
if RUN_FIXED_PARAMETERS or RUN_CALIBRATION:
    subprocess.run(command, check=True, cwd=ROOT)
else:
    print("Native execution is disabled. Command for a later explicit run:")
    print(subprocess.list2cmdline(command))


## 6. Read the outputs

Each run uses a new output directory. The daily table retains upstream observations, downstream simulation and observations, and quality flags. Reported flows in the table are cfs. Metrics use m3/s for MAE/RMSE/bias, dimensionless relative volume bias, NSE and KGE.

Read calibration and validation scores separately, with yearly scores and valid-day counts. The routing check independently reconstructs native convolutions and accounts for the tail beyond the final simulation date. Execution success and scientific acceptance are recorded separately.

In [ ]:
if (RUN_DIR / "report.json").is_file():
    result = json.loads((RUN_DIR / "report.json").read_text(encoding="utf-8"))
    print(json.dumps({key: result[key] for key in ("status", "metrics", "annual_metrics", "routing_check")}, indent=2))
else:
    print("No native results have been generated by this default walkthrough.")


## 7. Connect the later reservoir-operation comparison

After reviewing and freezing hydrology/routing, supply each operator's simulated reservoir outflow and the correctly partitioned local runoff. Compare all three operators over the same 2021–2025 dates at both gauges and at the reservoir. Observed White Haven flow is used for the preceding river/watershed test; the operator comparison uses the coupled simulation.

See the [English reference](../examples/fe_walter/downstream/validation.md) or [繁體中文說明](../examples/fe_walter/downstream/validation.zh-TW.md).